In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from IPython.display import display
import pickle

In [3]:
# load data
projectDir = Path.cwd().resolve().parent
dataPath = projectDir / "data" / "processed" / "matchupDiff_week5_features.csv"

matchupDiff = pd.read_csv(dataPath)
matchupDiff["y"] = matchupDiff["y"].astype(int)

In [4]:
# dropping any columns that could cause data leakage
leakCols = [
    "WScore", "LScore", "WTeamID", "LTeamID", "winnerTeamId",
    "team1Name", "team2Name", "Unnamed: 0"
]

idCols = ["team1Id", "team2Id"]

# all dropped columns
dropCols = ["Season", "y"] + leakCols + idCols

In [5]:
# train = 2003 - 2021, validation 2022, test 2023 - 2025
trainDf = matchupDiff[matchupDiff["Season"] <= 2021].copy()
valDf = matchupDiff[matchupDiff["Season"] == 2022].copy()
testDf = matchupDiff[matchupDiff["Season"].between(2023, 2025)].copy()

In [6]:
def buildXy(splitDf, featureCols=None):
    x = splitDf.drop(columns=dropCols, errors="ignore")
    x = x.select_dtypes(include=[np.number])

    if featureCols is None:
        featureCols = x.columns.tolist()
    else:
        x = x.reindex(columns=featureCols)

    y = splitDf["y"].astype(int)

    return x, y, featureCols


xTrain, yTrain, featureCols = buildXy(trainDf)
xVal, yVal, _ = buildXy(valDf, featureCols)
xTest, yTest, _ = buildXy(testDf, featureCols)

print(f"Train shape: {xTrain.shape}, Val shape: {xVal.shape}, Test shape: {xTest.shape}")

Train shape: (1162, 108), Val shape: (63, 108), Test shape: (128, 108)


In [7]:
def evaluateModel(pipeline, x, y, threshold=0.5):

    proba = pipeline.predict_proba(x)[:, 1]
    preds = (proba >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0)
    }

    cm = confusion_matrix(y, preds)

    cmDf = pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Pred 0", "Pred 1"]
    )

    return metrics, cmDf

In [8]:
paramGrid = {
    "nEstimators": [200, 300, 500],
    "maxDepth": [None, 10, 20],
    "minSamplesSplit": [2, 5],
    "maxFeatures": ["sqrt", "log2"],
    "classWeight": [None, "balanced"],
    "threshold": [0.5],
    "randomState": [226]
}

configs = list(ParameterGrid(paramGrid))
print(f"Total model configs: {len(configs)}")

Total model configs: 72


In [9]:
results = []

for idx, cfg in enumerate(configs):

    pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=cfg["nEstimators"],
            max_depth=cfg["maxDepth"],
            min_samples_split=cfg["minSamplesSplit"],
            max_features=cfg["maxFeatures"],
            class_weight=cfg["classWeight"],
            random_state=cfg["randomState"],
            n_jobs=-1
        ))
    ])

    pipe.fit(xTrain, yTrain)

    trainMetrics, _ = evaluateModel(pipe, xTrain, yTrain, cfg["threshold"])
    valMetrics, _ = evaluateModel(pipe, xVal, yVal, cfg["threshold"])

    results.append({
        "configId": idx,
        "model": str(cfg),
        "set": "train",
        **trainMetrics
    })

    results.append({
        "configId": idx,
        "model": str(cfg),
        "set": "val",
        **valMetrics
    })

resultsDf = pd.DataFrame(results)
resultsDf

,configId,model,set,accuracy,precision,recall
0,0,"{'classWeight': None, 'maxDepth': None, 'maxFe...",train,1.000000,1.000000,1.000
1,0,"{'classWeight': None, 'maxDepth': None, 'maxFe...",val,0.730159,0.734694,0.900
2,1,"{'classWeight': None, 'maxDepth': None, 'maxFe...",train,1.000000,1.000000,1.000
3,1,"{'classWeight': None, 'maxDepth': None, 'maxFe...",val,0.746032,0.740000,0.925
4,2,"{'classWeight': None, 'maxDepth': None, 'maxFe...",train,1.000000,1.000000,1.000
...,...,...,...,...,...,...
139,69,"{'classWeight': 'balanced', 'maxDepth': 20, 'm...",val,0.730159,0.709091,0.975
140,70,"{'classWeight': 'balanced', 'maxDepth': 20, 'm...",train,1.000000,1.000000,1.000
141,70,"{'classWeight': 'balanced', 'maxDepth': 20, 'm...",val,0.761905,0.735849,0.975
142,71,"{'classWeight': 'balanced', 'maxDepth': 20, 'm...",train,1.000000,1.000000,1.000


In [10]:
# top 3 by accuracy
valOnly = resultsDf[resultsDf["set"] == "val"].copy()

valOnly = valOnly.sort_values(
    ["accuracy", "recall", "precision"],
    ascending=False
)

top3Ids = valOnly.head(3)["configId"].tolist()

top3Results = resultsDf[resultsDf["configId"].isin(top3Ids)].copy()

top3Results = top3Results.sort_values(["configId", "set"])

display(top3Results)

# Compare validation metrics
top3ValCompare = top3Results[top3Results["set"] == "val"].copy()

top3ValCompare = top3ValCompare.sort_values(
    ["accuracy", "recall", "precision"],
    ascending=False
)

display(top3ValCompare)

,configId,model,set,accuracy,precision,recall
10,5,"{'classWeight': None, 'maxDepth': None, 'maxFe...",train,0.999139,0.998726,1.000
11,5,"{'classWeight': None, 'maxDepth': None, 'maxFe...",val,0.777778,0.750000,0.975
32,16,"{'classWeight': None, 'maxDepth': 10, 'maxFeat...",train,0.991394,0.987406,1.000
33,16,"{'classWeight': None, 'maxDepth': 10, 'maxFeat...",val,0.793651,0.764706,0.975
34,17,"{'classWeight': None, 'maxDepth': 10, 'maxFeat...",train,0.990534,0.986164,1.000
35,17,"{'classWeight': None, 'maxDepth': 10, 'maxFeat...",val,0.777778,0.750000,0.975


,configId,model,set,accuracy,precision,recall
33,16,"{'classWeight': None, 'maxDepth': 10, 'maxFeat...",val,0.793651,0.764706,0.975
11,5,"{'classWeight': None, 'maxDepth': None, 'maxFe...",val,0.777778,0.750000,0.975
35,17,"{'classWeight': None, 'maxDepth': 10, 'maxFeat...",val,0.777778,0.750000,0.975


In [11]:
# pick the winner on validation
valOnly = resultsDf[resultsDf["set"] == "val"].copy()

valOnly = valOnly.sort_values(
    ["accuracy", "recall", "precision"],
    ascending=False
)

bestRow = valOnly.iloc[0]
bestCfg = configs[int(bestRow["configId"])]

print("Best model (by validation accuracy, then recall, precision):")
print(bestRow)

Best model (by validation accuracy, then recall, precision):
configId                                                    16
model        {'classWeight': None, 'maxDepth': 10, 'maxFeat...
set                                                        val
accuracy                                              0.793651
precision                                             0.764706
recall                                                   0.975
Name: 33, dtype: object


In [12]:
# pipeline to find the best parameters based on accuracy
bestPipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=bestCfg["nEstimators"],
        max_depth=bestCfg["maxDepth"],
        min_samples_split=bestCfg["minSamplesSplit"],
        max_features=bestCfg["maxFeatures"],
        class_weight=bestCfg["classWeight"],
        random_state=bestCfg["randomState"],
        n_jobs=-1
    ))
])

bestPipe.fit(xTrain, yTrain)

trainMetrics, trainCm = evaluateModel(bestPipe, xTrain, yTrain, bestCfg["threshold"])
valMetrics, valCm = evaluateModel(bestPipe, xVal, yVal, bestCfg["threshold"])

print("Train metrics:", trainMetrics)
display(trainCm)

print("Validation metrics:", valMetrics)
display(valCm)

Train metrics: {'accuracy': 0.9913941480206541, 'precision': 0.9874055415617129, 'recall': 1.0}


,Pred 0,Pred 1
Actual 0,368,10
Actual 1,0,784


Validation metrics: {'accuracy': 0.7936507936507936, 'precision': 0.7647058823529411, 'recall': 0.975}


,Pred 0,Pred 1
Actual 0,11,12
Actual 1,1,39


In [13]:
# use best model on test
trainValDf = matchupDiff[matchupDiff["Season"] <= 2022].copy()

xTrainVal, yTrainVal, _ = buildXy(trainValDf, featureCols)

bestPipe.fit(xTrainVal, yTrainVal)

testMetrics, testCm = evaluateModel(bestPipe, xTest, yTest, bestCfg["threshold"])

print("Test metrics (2023-2025):", testMetrics)
display(testCm)

Test metrics (2023-2025): {'accuracy': 0.7109375, 'precision': 0.7264150943396226, 'recall': 0.9058823529411765}


,Pred 0,Pred 1
Actual 0,14,29
Actual 1,8,77


In [14]:
# exporting model to pickle
modelDir = projectDir / "models"
modelDir.mkdir(parents=True, exist_ok=True)

modelPath = modelDir / "rf_week7_best.pkl"

payload = {
    "model": bestPipe,
    "threshold": bestCfg["threshold"],
    "featureCols": featureCols,
    "params": bestCfg
}

with open(modelPath, "wb") as f:
    pickle.dump(payload, f)

In [15]:
# top 3 models to csv
processedDir = projectDir / "data" / "processed"
processedDir.mkdir(parents=True, exist_ok=True)

top3CsvPath = processedDir / "week7_top3_rf_models.csv"

top3Results.to_csv(top3CsvPath, index=False)

In [16]:
modelDir = projectDir / "models"
modelDir.mkdir(parents=True, exist_ok=True)

modelPath = modelDir / "rf_week7_best.pkl"
with open(modelPath, "wb") as f:
    pickle.dump(payload, f)